In [1]:
import json
import base64
from openai import OpenAI

In [2]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [7]:
SYSTEM_PROMPT_TOOLS = """\
You are evaluating answers about surgical tool counts. Compare two answers and decide if they express the SAME NUMBER of tools.

CRITICAL RULES:
1. Numbers match if they express the same quantity:
   - "0" matches "no tools" matches "i don't see any tools" matches "zero"
   - "1" matches "one tool" matches "one" matches "a single tool"
   - "2" matches "two tools" matches "two"

2. Match if both answers say the SAME COUNT:
   - "0" vs "i don't see any tools" → MATCH (both say zero)
   - "1" vs "one tool is operating" → MATCH (both say one)
   - "2" vs "Two tools" → MATCH (both say two)

3. NO_MATCH only if:
   - Different numbers: "0" vs "1", "1" vs "2"
   - One says tools exist, other says none: "1" vs "no tools"

4. Extract the NUMBER from verbose answers:
   - "i don't see any tools" → NUMBER: 0
   - "one tool is operating" → NUMBER: 1
   - "Two surgical instruments are visible" → NUMBER: 2

EXAMPLES:
Answer 1: "0" | Answer 2: "i don't see any tools" → MATCH (both say 0)
Answer 1: "1" | Answer 2: "one tool is operating" → MATCH (both say 1)
Answer 1: "2" | Answer 2: "Three tools" → NO_MATCH (2 vs 3)

Return ONLY:
MATCH
or
NO_MATCH
"""


In [8]:
def _parse_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [9]:
def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [12]:
def compare_answers(answer1: str, answer2: str) -> str:
    """Compare two answers and return whether they match in meaning.

    Returns: "MATCH" or "NO_MATCH"
    """

    user_text = (
        f"Answer 1: {answer1}\n"
        f"Answer 2: {answer2}"
    )

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=SYSTEM_PROMPT_TOOLS,
            input=user_text,
        )

        result = response.output_text.strip()

        # Extract MATCH or NO_MATCH from response
        if "MATCH" in result.upper():
            if "NO_MATCH" in result.upper():
                no_match_idx = result.upper().find("NO_MATCH")
                match_idx = result.upper().find("MATCH")
                if no_match_idx < match_idx:
                    return "NO_MATCH"
            return "MATCH"
        else:
            return "NO_MATCH"

    except Exception as e:
        print(f"Error comparing answers: {e}")
        return None

In [11]:
# Load the JSON
json_file = '/home/as5606/projects/Hulu-Med/hulumed-4B/accuracy_comparison.json'

with open(json_file, 'r') as f:
    data = json.load(f)

In [14]:
from tqdm import tqdm

#Filter only "how many tools are operating?" questions
tools_questions = [item for item in data if "how many tools are operating" in item.get('question', '').lower()]

print(f"Found {len(tools_questions)} 'how many tools' questions")

# Update matches for these questions
for item in tqdm(tools_questions):
    original_answer = item['original_answer']
    model_answer = item['model_answer']
    match_result = compare_answers(original_answer, model_answer)
    item['match'] = match_result

# Save back
with open(json_file, 'w') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated {len(tools_questions)} items in {json_file}")

Found 947 'how many tools' questions


100%|██████████| 947/947 [14:40<00:00,  1.08it/s]

✓ Updated 947 items in /home/as5606/projects/Hulu-Med/hulumed-4B/accuracy_comparison.json


In [15]:
# Load the JSON
json_file = '/home/as5606/projects/Hulu-Med/hulumed-7B/accuracy_comparison.json'

with open(json_file, 'r') as f:
    data = json.load(f)

from tqdm import tqdm

#Filter only "how many tools are operating?" questions
tools_questions = [item for item in data if "how many tools are operating" in item.get('question', '').lower()]

print(f"Found {len(tools_questions)} 'how many tools' questions")

# Update matches for these questions
for item in tqdm(tools_questions):
    original_answer = item['original_answer']
    model_answer = item['model_answer']
    match_result = compare_answers(original_answer, model_answer)
    item['match'] = match_result

# Save back
with open(json_file, 'w') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated {len(tools_questions)} items in {json_file}")

Found 947 'how many tools' questions


100%|██████████| 947/947 [14:56<00:00,  1.06it/s]  

✓ Updated 947 items in /home/as5606/projects/Hulu-Med/hulumed-7B/accuracy_comparison.json


In [16]:
# Load the JSON
json_file = '/home/as5606/projects/Hulu-Med/hulumed-14B/accuracy_comparison.json'

with open(json_file, 'r') as f:
    data = json.load(f)

from tqdm import tqdm

#Filter only "how many tools are operating?" questions
tools_questions = [item for item in data if "how many tools are operating" in item.get('question', '').lower()]

print(f"Found {len(tools_questions)} 'how many tools' questions")

# Update matches for these questions
for item in tqdm(tools_questions):
    original_answer = item['original_answer']
    model_answer = item['model_answer']
    match_result = compare_answers(original_answer, model_answer)
    item['match'] = match_result

# Save back
with open(json_file, 'w') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated {len(tools_questions)} items in {json_file}")

Found 947 'how many tools' questions


100%|██████████| 947/947 [14:50<00:00,  1.06it/s]


✓ Updated 947 items in /home/as5606/projects/Hulu-Med/hulumed-14B/accuracy_comparison.json
